# P14 — Toolformer: los modelos de lenguaje pueden enseñarse a sí mismos a usar herramientas

## 1. Título y paper

**Paper:** *Toolformer: Language Models Can Teach Themselves to Use Tools*  
**Autoría:** Timo Schick, Jane Dwivedi-Yu, Roberto Dessì, Roberta Raileanu, y otros  
**Año y venue:** 2023 · arXiv:2302.04761 · NeurIPS 2023  
**Nivel:** L3 · **Motor:** `toolformer`  
**Ficha completa:** [`P14_toolformer`](../../papers/foundational/P14_toolformer/README.md)

**Hito:** El uso de herramientas se aprende de forma autosupervisada: el criterio de utilidad es la propia pérdida del modelo.

- [arXiv:2302.04761](https://arxiv.org/abs/2302.04761)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Enseñar a un modelo a llamar APIs requería datos anotados por humanos, caros y limitados a las herramientas anotadas.
2. Ejecutar una implementación mínima de la propuesta: Generar llamadas candidatas, ejecutarlas y conservar solo las que reducen la pérdida de predecir el texto siguiente; reentrenar con ese corpus filtrado.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P13


## 4. Intuición

En vez de que un humano anote «aquí deberías usar la calculadora», el modelo prueba a llamarla en muchos sitios, se queda con las llamadas que le ayudaron a predecir mejor lo que venía después, y aprende de ese corpus filtrado por sí mismo.


## 5. Concepto mínimo

```text
1. muestrear posiciones y llamadas candidatas
2. ejecutar la API y obtener el resultado r
3. conservar la llamada si  L(con resultado) < L(sin llamada) − τ
4. reentrenar el modelo sobre el texto con las llamadas conservadas
```

El criterio de utilidad es la **pérdida del propio modelo**: nadie etiqueta nada.


## 6. Código explicado

El motor aplica el filtro sobre tres candidatas: una útil, una marginal y una absurda.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('toolformer', seed=7)['result']
print('umbral τ =', r['umbral'], '\n')
for fila in r['candidatos_evaluados']:
    marca = '✅' if fila['se_conserva'] else '❌'
    print(f"{marca} Δpérdida={fila['reduccion_de_perdida']:+.2f} · {fila['llamada']}")

## 7. Predicción antes de ejecutar

1. ¿Se conservará la llamada al buscador que solo reduce la pérdida en 0,04?
2. ¿Qué signo tendrá Δpérdida en la llamada absurda?
3. Si bajas τ a 0,01, ¿qué problema aparece?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
candidatos = r['candidatos_evaluados']
for tau in (0.01, 0.30, 1.00, 3.00):
    conservados = [c['llamada'] for c in candidatos if c['reduccion_de_perdida'] > tau]
    print(f'τ={tau:<5} → {len(conservados)} llamadas conservadas: {conservados}')

## 9. Salida interpretable

τ es el mando entre dos fallos opuestos: **τ bajo** conserva llamadas inútiles (el modelo aprende a llamar herramientas todo el rato, con su coste y su latencia); **τ alto** descarta llamadas útiles (el modelo vuelve a inventar los cálculos). No existe un τ universal.


## 10. Comentario pedagógico

El método enseña *cuándo* llamar, no garantiza que la herramienta acierte. Un Toolformer conectado a una API que devuelve basura aprenderá a llamarla con confianza. La calidad de la herramienta es un supuesto del método, no un resultado.


## 11. Error o anti-patrón deliberado

Anti-patrón: medir el éxito por «número de llamadas a herramientas» como si más fuera mejor.


In [ ]:
metrica_mala = {'llamadas_por_respuesta': 7, 'conclusion': 'el agente usa mucho las herramientas'}
show(metrica_mala)
print('→ 7 llamadas pueden significar competencia… o un bucle caro que no converge.')

## 12. Corrección

Métricas que sí informan: utilidad marginal por llamada, coste y tasa de llamadas descartadas.


In [ ]:
util = [c for c in candidatos if c['se_conserva']]
metricas = {
    'candidatas_evaluadas': len(candidatos),
    'conservadas': len(util),
    'tasa_de_descarte': round(1 - len(util) / len(candidatos), 3),
    'reduccion_media_de_perdida': round(sum(c['reduccion_de_perdida'] for c in util) / max(len(util), 1), 3),
}
show(metricas)

## 13. Desafío guiado

Añade una candidata nueva con Δpérdida negativa grande y comprueba que el filtro la rechaza sin intervención humana.


In [ ]:
nueva = {'texto': 'Erase una vez un bosque.', 'llamada': '[Calc(erase) -> error]',
         'loss_sin': 0.80, 'loss_con': 2.40}
delta = nueva['loss_sin'] - nueva['loss_con']
print(f"Δpérdida = {delta:+.2f} → se conserva: {delta > r['umbral']}")

## 14. Desafío autónomo

Con un modelo abierto pequeño y un calculador local, implementa el filtrado real por perplejidad sobre 200 frases con operaciones aritméticas. Reporta cuántas llamadas sobreviven y cómo cambia el resultado al variar τ.


## 15. Evidencia de aprendizaje

Guarda la tabla de candidatas con su Δpérdida, el barrido de τ y las métricas que sustituyen al conteo bruto de llamadas.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P14_toolformer/README.md) · evaluación formal: [`assessments/papers/P14_toolformer.md`](../../assessments/papers/P14_toolformer.md)


## 16. Cierre

El uso de herramientas ya se autosupervisa. Volvemos a la alineación: ¿se puede conseguir el efecto de RLHF sin su maquinaria?


## 17. Conexión con el siguiente hito

- P16

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
